# mlsim — Hardware Benchmark (T4)

Measures real matmul runtimes across dtypes and matrix sizes on a Colab T4 GPU.
Results calibrate the cycle-approximate simulator's quantization crossover model.

**Runtime:** ~10-15 minutes. **Output:** `results.csv`

In [ ]:
import torch, csv
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
SIZES        = [32, 64, 128, 256, 512, 1024, 2048, 4096]
WARMUP       = 20
ITERS        = 100
DTYPE_MAP    = {'fp32': torch.float32, 'fp16': torch.float16, 'bf16': torch.bfloat16}
DTYPE_BYTES  = {'fp32': 4, 'fp16': 2, 'bf16': 2, 'int8': 1}

In [ ]:
def bench_fp(size, dtype):
    A = torch.randn(size, size, dtype=dtype, device=device)
    B = torch.randn(size, size, dtype=dtype, device=device)
    for _ in range(WARMUP): torch.mm(A, B)
    torch.cuda.synchronize()
    s = torch.cuda.Event(enable_timing=True)
    e = torch.cuda.Event(enable_timing=True)
    s.record()
    for _ in range(ITERS): torch.mm(A, B)
    e.record()
    torch.cuda.synchronize()
    return s.elapsed_time(e) / ITERS

def bench_int8(size):
    A = torch.randint(-128, 127, (size, size), dtype=torch.int8, device=device)
    B = torch.randint(-128, 127, (size, size), dtype=torch.int8, device=device)
    try:
        for _ in range(WARMUP): torch._int_mm(A, B)
        torch.cuda.synchronize()
        s = torch.cuda.Event(enable_timing=True)
        e = torch.cuda.Event(enable_timing=True)
        s.record()
        for _ in range(ITERS): torch._int_mm(A, B)
        e.record()
        torch.cuda.synchronize()
        return s.elapsed_time(e) / ITERS
    except Exception as ex:
        print(f'  int8 skipped at {size}: {ex}')
        return None

def arith_intensity(size, db):
    return (2 * size**3) / (db * 3 * size**2)

In [ ]:
rows = []
for size in SIZES:
    print(f'\nSize {size}x{size}')
    for dname, tdtype in DTYPE_MAP.items():
        ms     = bench_fp(size, tdtype)
        flops  = 2 * size**3
        tflops = flops / (ms * 1e-3) / 1e12
        intens = arith_intensity(size, DTYPE_BYTES[dname])
        print(f'  {dname:5s}  {ms:.4f} ms  {tflops:.3f} TFLOPS  AI={intens:.1f}')
        rows.append({'hardware':'GPU (T4)','workload':f'MatMul {size}x{size}x{size}',
                     'matrix_size':size,'dtype':dname,'runtime_ms':round(ms,6),
                     'tflops':round(tflops,4),'arithmetic_intensity':round(intens,2),
                     'flops':flops,'bytes_accessed':DTYPE_BYTES[dname]*3*size**2})
    ms8 = bench_int8(size)
    if ms8 is not None:
        flops  = 2 * size**3
        tflops = flops / (ms8 * 1e-3) / 1e12
        intens = arith_intensity(size, 1)
        print(f'  int8   {ms8:.4f} ms  {tflops:.3f} TFLOPS  AI={intens:.1f}')
        rows.append({'hardware':'GPU (T4)','workload':f'MatMul {size}x{size}x{size}',
                     'matrix_size':size,'dtype':'int8','runtime_ms':round(ms8,6),
                     'tflops':round(tflops,4),'arithmetic_intensity':round(intens,2),
                     'flops':flops,'bytes_accessed':3*size**2})
print('\nDone.')

In [ ]:
import matplotlib.pyplot as plt
colors = {'fp32':'#2D6BE4','fp16':'#27AE60','bf16':'#F39C12','int8':'#E84545'}
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
for dn in ['fp32','fp16','bf16','int8']:
    sub = [r for r in rows if r['dtype']==dn]
    if not sub: continue
    ax1.plot([r['matrix_size'] for r in sub],[r['runtime_ms'] for r in sub],
             marker='o',label=dn,color=colors[dn],linewidth=2)
    ax2.scatter([r['arithmetic_intensity'] for r in sub],[r['tflops'] for r in sub],
                label=dn,color=colors[dn],s=60,zorder=4)
    ax2.plot([r['arithmetic_intensity'] for r in sub],[r['tflops'] for r in sub],
             color=colors[dn],linewidth=1.5,alpha=0.6)
ax1.set_xlabel('Matrix size'); ax1.set_ylabel('Runtime (ms)'); ax1.set_yscale('log')
ax1.set_title('Runtime vs matrix size'); ax1.legend(); ax1.grid(True,alpha=0.3)
ax2.set_xlabel('Arithmetic intensity'); ax2.set_ylabel('TFLOPS'); ax2.set_xscale('log')
ax2.set_title('TFLOPS vs arithmetic intensity'); ax2.legend(); ax2.grid(True,alpha=0.3)
plt.suptitle('T4 Benchmark Results',fontsize=13,fontweight='bold')
plt.tight_layout()
plt.savefig('benchmark_preview.png',dpi=150,bbox_inches='tight')
plt.show()

In [ ]:
fields = ['hardware','workload','matrix_size','dtype','runtime_ms',
          'tflops','arithmetic_intensity','flops','bytes_accessed']
with open('results.csv','w',newline='') as f:
    w = csv.DictWriter(f, fieldnames=fields)
    w.writeheader(); w.writerows(rows)
print(f'Saved {len(rows)} rows to results.csv')

In [ ]:
from google.colab import files
files.download('results.csv')
files.download('benchmark_preview.png')